In [18]:
from pathlib import Path
from os import path as osp
import numpy as np
from nuscenes.nuscenes import NuScenes
from nuscenes.map_expansion.map_api import NuScenesMap, NuScenesMapExplorer
from shapely.geometry import Polygon, MultiPolygon
from copy import deepcopy
from nusc_to_wgs_conversion import convert_city_coords_to_wgs84
import requests

In [ ]:
overpass_useragent = 'FZI'
overpass_headers = {
    'Connection': 'keep-alive',
    'sec-ch-ua': '"Google Chrome 80"',
    'Accept': '*/*',
    'Sec-Fetch-Dest': 'empty',
    'User-Agent': overpass_useragent,
    'Content-Type': 'application/x-www-form-urlencoded; charset=UTF-8',
    'Origin': 'https://overpass-turbo.eu',
    'Sec-Fetch-Site': 'cross-site',
    'Sec-Fetch-Mode': 'cors',
    'Referer': 'https://overpass-turbo.eu/',
    'Accept-Language': '',
    'dnt': '1',
}

nusc_root_path = "/tmp/nuscenes"

out_path = "out"

nusc = NuScenes(version='v1.0-trainval', dataroot=nusc_root_path, verbose=True)
MAPS = ['boston-seaport', 'singapore-hollandvillage',
                 'singapore-onenorth', 'singapore-queenstown']
nusc_maps = {}
map_explorer = {}
for loc in MAPS:
    nusc_maps[loc] = NuScenesMap(dataroot=nusc_root_path, map_name=loc)
    map_explorer[loc] = NuScenesMapExplorer(nusc_maps[loc])

In [ ]:
import nusc_to_wgs_conversion as nusc_to_wgs
from importlib import reload
reload(nusc_to_wgs)

for loc in MAPS:
    print(loc)
    xmin, ymin, xmax, ymax = float("inf"), float("inf"), float("-inf"), float("-inf")
    for area in nusc_maps[loc].drivable_area:
        xmin_area, ymin_area, xmax_area, ymax_area = nusc_maps[loc].get_bounds('drivable_area', area['token'])
        xmin = min(xmin, xmin_area)
        ymin = min(ymin, ymin_area)
        xmax = max(xmax, xmax_area)
        ymax = max(ymax, ymax_area)

    print(np.array([[xmin - 100, ymin - 100], [xmax + 100, ymax + 100]]))
    bounds_latlon = nusc_to_wgs.convert_city_coords_to_wgs84(np.array([[xmin - 100, ymin - 100], [xmax + 100, ymax + 100]]), loc)
    print(bounds_latlon)

    query_str = "(node({0},{1},{2},{3}); rel(bn)->.x; way({0},{1},{2},{3}); node(w)->.x; rel(bw);); out meta;".format(bounds_latlon[0, 0], bounds_latlon[0, 1], bounds_latlon[1, 0], bounds_latlon[1, 1])
    data = {
      'data': query_str
    }
    response = requests.post('https://overpass-api.de/api/interpreter', headers=overpass_headers, data=data)
    save_path = osp.join(out_path)
    Path(save_path).mkdir(parents=True, exist_ok=True)
    with open(osp.join(save_path, loc + '.osm'), 'w') as f:
        f.write(response.text)